# VerinBirds — Этап 4: CNN transfer learning (ResNet18) — версия для KAGGLE

Запускать в **Kaggle Notebooks** (у Kaggle отдельная бесплатная квота GPU, ~30 ч/нед).

## Что сделать перед запуском (важно, 3 настройки справа)
1. **Данные:** правая панель → **Add Input** → найди датасет `chicken-egg-analysis-dataset`
   (автор **hlnkmb**) → **Add**. Он появится в `/kaggle/input/`.
2. **GPU:** правая панель → **Session options / Accelerator** → выбери **GPU T4 x2** (или P100).
3. **Интернет:** правая панель → **Internet → On** (нужно для скачивания предобученных весов
   ResNet18). Требует привязанного телефона в аккаунте Kaggle — это бесплатно и делается один раз.

Потом: **Run All**. В конце файлы `verinbirds_resnet18.pt` и `metrics.json` появятся в
`/kaggle/working/` (правая панель, вкладка **Output** / **Data → output**) — оттуда скачаешь.

**Честная рамка:** высокая точность на курином test ожидаема и мало значит —
классы разделимы по цвету/источнику (baseline дал 99.8%). Настоящая проверка — перенос на
перепелов (Этап 6, локально). Здесь цель — получить рабочие веса.


In [ ]:
# Ячейка 1 — окружение и GPU
import torch, torchvision, sys
print("python:", sys.version.split()[0], "| torch:", torch.__version__, "| torchvision:", torchvision.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("!!! GPU не подключён. Справа: Accelerator -> GPU T4, затем Run All заново.")


In [ ]:
# Ячейка 2 — данные: датасет подключён через Add Input, ищем его в /kaggle/input
import glob
from pathlib import Path
print("Подключённые входы:", glob.glob('/kaggle/input/*'))
EXT = {'.jpg', '.jpeg', '.png'}
imgs = [p for p in Path('/kaggle/input').rglob('*') if p.suffix.lower() in EXT]
print("изображений найдено:", len(imgs))
assert len(imgs) > 3000, ("Датасет не подключён. Справа Add Input -> "
    "'chicken-egg-analysis-dataset' (автор hlnkmb) -> Add, потом перезапусти ячейку.")
DATA_ROOT = Path('/kaggle/input')


In [ ]:
# Ячейка 3 — конфиг + препроцессинг (letterbox как в ml/src/preprocess.py) + аугментации
import random, numpy as np
from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IMG_SIZE = 224
CLASSES = ["fertile", "infertile", "dead"]
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
IMAGENET_MEAN = (0.485, 0.456, 0.406); IMAGENET_STD = (0.229, 0.224, 0.225)

class LetterboxResize:
    "Вписать в квадрат с сохранением пропорций + чёрный паддинг (== preprocess.py)"
    def __init__(self, size=IMG_SIZE): self.size = size
    def __call__(self, im):
        im = ImageOps.exif_transpose(im).convert("RGB")
        w, h = im.size; s = self.size / max(w, h)
        nw, nh = max(1, round(w*s)), max(1, round(h*s))
        im = im.resize((nw, nh), Image.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), (0, 0, 0))
        canvas.paste(im, ((self.size-nw)//2, (self.size-nh)//2))
        return canvas

train_tf = T.Compose([LetterboxResize(), T.RandomHorizontalFlip(),
                      T.RandomRotation(15, fill=0),
                      T.ColorJitter(brightness=0.2, contrast=0.2),
                      T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf  = T.Compose([LetterboxResize(), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def cls_of(name):
    n = name.lower()
    return "infertile" if "infertile" in n else "fertile" if "fertile" in n else "dead" if "dead" in n else None

class EggDS(Dataset):
    def __init__(self, files, tf): self.files = files; self.tf = tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        p = self.files[i]
        with Image.open(p) as im: x = self.tf(im)
        return x, CLS2IDX[cls_of(Path(p).name)]


In [ ]:
# Ячейка 4 — сплиты: train/val из training/, test из testing/ (по пути файла)
all_imgs = [p for p in DATA_ROOT.rglob('*') if p.suffix.lower() in EXT and cls_of(p.name)]
train_files = [str(p) for p in all_imgs if 'training' in str(p).lower()]
test_files  = [str(p) for p in all_imgs if 'testing'  in str(p).lower()]
assert train_files and test_files, "Не найдены папки training/testing — проверь структуру датасета"

from sklearn.model_selection import train_test_split
y_tr = [cls_of(Path(p).name) for p in train_files]
tr_files, val_files = train_test_split(train_files, test_size=0.15, stratify=y_tr, random_state=SEED)

from collections import Counter
print("train:", len(tr_files), Counter(cls_of(Path(p).name) for p in tr_files))
print("val:  ", len(val_files), Counter(cls_of(Path(p).name) for p in val_files))
print("test: ", len(test_files), Counter(cls_of(Path(p).name) for p in test_files))

BATCH = 32
dl_train = DataLoader(EggDS(tr_files, train_tf), batch_size=BATCH, shuffle=True, num_workers=2)
dl_val   = DataLoader(EggDS(val_files, eval_tf), batch_size=BATCH, shuffle=False, num_workers=2)
dl_test  = DataLoader(EggDS(test_files, eval_tf), batch_size=BATCH, shuffle=False, num_workers=2)


In [ ]:
# Ячейка 5 — модель: ResNet18 (ImageNet), заморозка backbone, учим layer4 + голову
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)   # нужен Internet=On
for p in model.parameters(): p.requires_grad = False
for p in model.layer4.parameters(): p.requires_grad = True   # хочешь быстрее на CPU — закомментируй
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(DEVICE)

opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
crit = nn.CrossEntropyLoss()
print("обучаемых тензоров:", sum(p.requires_grad for p in model.parameters()))


In [ ]:
# Ячейка 6 — обучение с ранней остановкой по val loss
import copy
def run_epoch(dl, train):
    model.train() if train else model.eval()
    tot, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train: opt.zero_grad()
            out = model(x); loss = crit(out, y)
            if train: loss.backward(); opt.step()
            loss_sum += loss.item()*x.size(0); tot += x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return loss_sum/tot, correct/tot

MAX_EPOCHS, PATIENCE = 15, 3
best_val, best_state, wait = 1e9, None, 0
for ep in range(1, MAX_EPOCHS+1):
    tl, ta = run_epoch(dl_train, True)
    vl, va = run_epoch(dl_val, False)
    print(f"epoch {ep:2d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")
    if vl < best_val - 1e-4:
        best_val, best_state, wait = vl, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= PATIENCE: print(f"ранняя остановка на эпохе {ep}"); break
model.load_state_dict(best_state)
print("лучшие веса загружены, val loss =", round(best_val,4))


In [ ]:
# Ячейка 7 — честная оценка на test
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

model.eval(); y_true, y_pred = [], []
with torch.no_grad():
    for x, y in dl_test:
        y_pred += model(x.to(DEVICE)).argmax(1).cpu().tolist(); y_true += y.tolist()

acc = accuracy_score(y_true, y_pred)
print("TEST accuracy:", round(acc, 4))
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,4.4)); im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("предсказано"); ax.set_ylabel("истина"); ax.set_title(f"ResNet18 test acc={acc:.3f}")
for i in range(3):
    for j in range(3): ax.text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("/kaggle/working/confusion_matrix_cnn.png", dpi=120); plt.show()


In [ ]:
# Ячейка 8 — сохранить веса + метрики в /kaggle/working (скачаешь из вкладки Output)
import json
from sklearn.metrics import recall_score
metrics = {
    "model": "resnet18_finetune_layer4",
    "test_accuracy": round(float(acc), 4),
    "per_class_recall": {c: round(float(r), 4) for c, r in zip(CLASSES, recall_score(y_true, y_pred, average=None))},
    "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    "classes": CLASSES, "img_size": IMG_SIZE, "seed": SEED,
    "n_train": len(tr_files), "n_val": len(val_files), "n_test": len(test_files),
    "note": "Обучено на КУРИНОМ датасете. Высокая точность ожидаема (артефакты источника). Реальная проверка — перепела, Этап 6."
}
with open("/kaggle/working/metrics.json", "w") as f: json.dump(metrics, f, ensure_ascii=False, indent=2)
torch.save(model.state_dict(), "/kaggle/working/verinbirds_resnet18.pt")
print("Сохранено в /kaggle/working/:  verinbirds_resnet18.pt, metrics.json, confusion_matrix_cnn.png\n")
print(json.dumps(metrics, ensure_ascii=False, indent=2))


## Готово

Файлы в `/kaggle/working/` (правая панель → вкладка **Output**, либо после Save Version — в Output ноутбука):
- `verinbirds_resnet18.pt` — веса. Скачай и положи в репозиторий `ml/models/`.
- `metrics.json` — метрики (для сравнения с baseline).
- `confusion_matrix_cnn.png` — картинка.

Напоминание: цель — рабочие веса, а не «победить baseline на курах».
Кто лучше — решит перенос на перепелов (Этап 6).